In [ ]:
import sys

sys.path.append("..")

import glob
import os

import pandas as pd

from nnspike.constants import ROI_CNN
from nnspike.data import (
    augment_dataset,
    create_label_dataframe,
    set_spike_status,
    sort_by_frames_number,
)
from notebooks.utils import extract_frames_from_avi_files, get_all_avi_files

course = "right"  # "right" or "left"

## Extract Frames

In [4]:
# Get all AVI files with their timestamps
avi_files_with_timestamps = get_all_avi_files(
    directory_path="../storage/videos/", filter_timestamp="20250824*"
)
avi_files_with_timestamps

In [ ]:
# Extract frames from all AVI files
if avi_files_with_timestamps:
    print("\nStarting frame extraction...")
    output_dirs_with_timestamps = extract_frames_from_avi_files(
        avi_files_with_timestamps, base_output_dir="../storage/frames/"
    )
    print("\nFrame extraction completed!")
    print(
        f"Successfully created {len(output_dirs_with_timestamps)} output directories:"
    )
    for output_dir, timestamp in output_dirs_with_timestamps:
        print(f"  - {output_dir} (timestamp: {timestamp})")
else:
    print("No AVI files found to process.")
    output_dirs_with_timestamps = []

output_dirs_with_timestamps

In [ ]:
for output_dir, timestamp in output_dirs_with_timestamps:
    print(f"Output directory: {output_dir} (timestamp: {timestamp})")

    label_df = create_label_dataframe(output_dir + "/*", course)
    label_df = sort_by_frames_number(label_df)

    status_df = pd.read_csv(f"../storage/sensor_data/{timestamp}_sensor_log.csv")
    df = set_spike_status(label_df, status_df)

    # Export to a csv file
    df.to_csv(f"../storage/labels/{timestamp}_label.csv", index=False)

## Data Augmentation

In [ ]:
# Define the directory path
directory_path = "../storage/labels"  # Replace with your actual directory path

# Pattern to match all files in the directory
# '*' matches any file or directory name
all_files_pattern = os.path.join(directory_path, "*")

# Get a list of all files and directories
all_entries = glob.glob(all_files_pattern)

# Filter out directories to show only files
raw_csv_labels = [entry for entry in all_entries if os.path.isfile(entry)]
raw_csv_labels

In [ ]:
raw_csv_labels = [
    "../storage/labels\\20250705150757_label.csv",
    "../storage/labels\\20250705153842_label.csv",
]

AUG_FLAG = "aug1"
csv_labels = [path.replace("\\", "/") for path in raw_csv_labels]

for csv_label in csv_labels:
    df = pd.read_csv(csv_label)
    df = df[(df["use"] == True)]
    timestamp = csv_label.split("/")[-1][:14]
    df = augment_dataset(df, 0.5, f"../storage/frames/{timestamp}_{AUG_FLAG}")
    df.to_csv(f"../storage/labels/{timestamp}_{AUG_FLAG}_label.csv", index=False)

## Train on AWS SageMaker 

### Download file from s3

In [ ]:
import boto3

# Create an S3 client
s3_client = boto3.client("s3")

# Define your S3 bucket name, object key, and local file path
bucket_name = "aws-bucket-name"
s3_object_key = "to/your/s3/key/file.zip"
local_file_path = "to/your/local/frames.zip"

try:
    s3_client.download_file(bucket_name, s3_object_key, local_file_path)
    print(f"File '{s3_object_key}' downloaded successfully to '{local_file_path}'")
except Exception as e:
    print(f"Error downloading file: {e}")

### Extract zip

In [ ]:
from zipfile import ZipFile

# Specify the name of your zip file
zip_file_name = "to/your/local/frames.zip"

try:
    # Open the zip file in read mode ('r')
    with ZipFile(local_file_path, "r") as zip_ref:
        # Extract all contents to the current working directory
        zip_ref.extractall("../storage/frames/")
    print(f"Successfully extracted '{zip_file_name}' to the current directory.")
except FileNotFoundError:
    print(f"Error: The file '{zip_file_name}' was not found.")
except Exception as e:
    print(f"An error occurred during extraction: {e}")